In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
def ler_ultima_particao_tabela_spark(spark, source_table):
  """
  Essa função ler a ultima partição das Tabelas no formato delta baaseado na coluna de data_processamento
  """
  try: 
    # Mais performatica para pegar os metadados
    show_partitions_df = spark.sql(f"SHOW PARTITIONS {source_table}")
    # maior partição da data_processamento
    max_partition = show_partitions_df.agg(f.max("data_processamento")).collect()[0][0]
    print(f"partição maxima {max_partition}")

    # pegar o dataframe com maior partição
    return spark.table(f"{source_table}")\
                      .filter(f.col("data_processamento") == max_partition)
  except Exception as e:
    print(f"Erro ao ler o caminho {source_table}: {e}")
    return None


### 1. Indicadores Economicos

In [0]:
silver_path_indicadores = "workspace.case_spark_cvm.silver_dados_indicadores_economicos"

df_silver_indicadores = ler_ultima_particao_tabela_spark(spark, silver_path_indicadores)

In [0]:
display(df_silver_indicadores)

### 2. Fundos Diarios

In [0]:
silver_table_cvm_day = "workspace.case_spark_cvm.silver_cvm_fundos_diario"

df_cvm_fundos_diario_silver = ler_ultima_particao_tabela_spark(spark, silver_table_cvm_day)

In [0]:
df_cvm_fundos_diario_silver.columns

In [0]:
gold_fato_diario = df_cvm_fundos_diario_silver\
    .select("cnpj_fundo_classe", "dt_comptc", "vl_quota", "vl_total", "vl_patrim_liq", "captc_dia", "resg_dia", "nr_cotst")

### 3. Registros Classes

In [0]:
silver_table_classe_cvm = "workspace.case_spark_cvm.silver_registro_classe_cvm"

df_registro_classe_cvm_silver = ler_ultima_particao_tabela_spark(spark, silver_table_classe_cvm)

In [0]:
df_registro_classe_cvm_silver = df_registro_classe_cvm_silver.select("cnpj_classe", "indicador_desempenho", "tipo_classe")

In [0]:
df_registro_classe_cvm_silver = df_registro_classe_cvm_silver\
.withColumn(
    "benchmark_normalizado",
    f.when(f.col("indicador_desempenho").isin("DI de um dia", "Taxa Anbid", "Taxa Básica Financeira", "Taxa Anbid", ), "CDI")\
    .when(f.col("indicador_desempenho").contains("Andima"), "CDI")\
    .when(f.col("indicador_desempenho").contains("Anbid"), "CDI")\
    .when(f.col("indicador_desempenho").isin("Taxa Selic"), "Selic")\
    .when(f.col("indicador_desempenho").isin("Ibovespa", "IBrX", "IBrX-50"), "Ibovespa")\
    .when(
        f.col("indicador_desempenho")\
        .isin(
            "Índice de Preços ao Consumidor Amplo (IPCA/IBGE)",
            "Índice Nacional de Preços ao Consumidor (INPC/IBGE)",
            "Índice de preços"
            ), "IPCA")\
    .when(
        f.col("indicador_desempenho")\
            .isin("Não se aplica", "OUTROS") | 
        f.col("indicador_desempenho").isNull(), 
        "SEM_BENCHMARK")
    .otherwise("NAO_DISPONIVEL")
)

### 4. Joins e Filtros

Os Fundos Imobiliarios  não entram na tabela **Fato Dário** devido aos FIIs terem um comportamento diferentes dos outros fundos

In [0]:
gold_fato_diario = gold_fato_diario\
    .join(
        df_silver_indicadores,
        gold_fato_diario.dt_comptc == df_silver_indicadores.data
    )\
    .join(
        df_registro_classe_cvm_silver,
        gold_fato_diario.cnpj_fundo_classe == df_registro_classe_cvm_silver.cnpj_classe
    )\
    .filter(f.col("tipo_classe") != "Classes de Cotas de Fundos FII")\
    .withColumnRenamed("valor_selic", "selic_anual")\
    .withColumnRenamed("valor_cdi", "cdi_diario")\
    .withColumn("selic_diaria", f.pow(1 + (f.col("selic_anual") / 100), 1/252) - 1)\
    .withColumn("ipca_diario", f.pow(1 + (f.col("ipca_mensal") / 100), 1/21) - 1)
    

In [0]:
display(gold_fato_diario)

### 5. Criação de Novas Features

In [0]:
window_spec = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc")

window_spec_first = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

window_inicio = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(Window.unboundedPreceding, 0)

window_data_inicio = Window.partitionBy("cnpj_fundo_classe")

window_21d = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(-20, 0) # janela 1 mês 

window_63d = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(-62, 0) # janela 3 mês 

window_126d = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(-125, 0) # janela 6 mês 

window_252d = Window.partitionBy("cnpj_fundo_classe").orderBy("dt_comptc").rowsBetween(-251, 0) # janela 1 ano 

gold_fato_diario = gold_fato_diario\
    .withColumn(
        "retorno_diario",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 1).over(window_spec) ) - 1 # retorno do dia em %
    )\
    .withColumn(
        "retorno_21d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 21).over(window_spec) ) - 1 # retorno ~1 mês
    )\
    .withColumn(
        "retorno_63d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 63).over(window_spec) ) - 1 # retorno ~3 meses
    )\
    .withColumn(
        "retorno_126d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 126).over(window_spec) ) - 1 # retorno ~6 meses
    )\
    .withColumn(
        "retorno_252d",
        f.try_divide(f.col("vl_quota"), f.lag("vl_quota", 252).over(window_spec) ) - 1 # retorno ~1 ano
    )\
    .withColumn(
        "retorno_inicio",
        f.try_divide(f.col("vl_quota"), f.first("vl_quota").over(window_spec_first) ) - 1 # retorno desde o início
    )\
    .withColumn(
        "captacao_liquida_dia",
         f.col("captc_dia") - f.col("resg_dia") # captação líquida diária
    )\
    .withColumn(
        "captacao_liquida_21d",
        f.sum("captacao_liquida_dia").over(window_21d) # captação líquida 1 mês
    )\
    .withColumn(
        "captacao_liquida_252d",
        f.sum("captacao_liquida_dia").over(window_252d) # captação líquida 1 ano
    )\
    .withColumn(
        "variacao_cotistas",
        f.try_divide(f.col("nr_cotst"), f.lag("nr_cotst", 1).over(window_spec) ) - 1 # variação diária de cotistas
    )\
    .withColumn(
        "volatilidade_21d",
        f.stddev("retorno_diario").over(window_21d) * f.sqrt(f.lit(252)) # vol anualizada 1 mês
    )\
    .withColumn(
        "volatilidade_63d",
        f.stddev("retorno_diario").over(window_63d) * f.sqrt(f.lit(252)) # vol anualizada 3 meses
    )\
    .withColumn(
        "volatilidade_252d",
        f.stddev("retorno_diario").over(window_252d) * f.sqrt(f.lit(252)) # vol anualizada 1 ano
    )\
    .withColumn(
        "max_quota_historico",
        f.max("vl_quota").over(window_inicio) # pico histórico da cota
    )\
    .withColumn(
        "drawdown",
        f.try_divide(f.col("vl_quota"), f.col("max_quota_historico") ) -1 # queda em relação ao pico
    )\
    .withColumn(
        "drawdown_maximo_252d",
        f.min("drawdown").over(window_252d) # pior drawdown em 1 ano
    )\
    .withColumn(
        "drawdown_maximo_historico", 
        f.min("drawdown").over(window_inicio)
    )\
    .withColumn(
        "retorno_negativo_diario",
        f.when(f.col("retorno_diario") < 0, f.col("retorno_diario")).otherwise(0) # só retornos negativos
    )\
    .withColumn(
        "downside_deviation_252d",
        f.stddev("retorno_negativo_diario").over(window_252d) * f.sqrt(f.lit(252)) # desvio negativo anualizado
    )\
    .withColumn(
        "var_95_252d",
        f.percentile_approx("retorno_diario", 0.05).over(window_252d) # Sortino 1 ano
    )\
    .withColumn(
        "data_inicio",
        f.min("dt_comptc").over(window_data_inicio) # VaR 95% histórico
    )\
    .withColumn(
        "flag_fundo_novo",
         f.when(f.datediff(f.col("dt_comptc"), f.col("data_inicio")) < 90, "S").otherwise("N") # S/N — fundo com menos de 90 dias
    )\
    .withColumn(
        "flag_resgate_consistente",
        f.when(f.col("captacao_liquida_21d") < 0, "S").otherwise("N") # S/N — saída de dinheiro no mês
    )\
    .withColumn(
        "cdi_acum_21d", 
        f.try_divide(f.col("indice_cdi"), f.lag("indice_cdi", 21).over(window_spec)) - 1
    ) \
    .withColumn(
        "selic_acum_21d", 
        f.try_divide(f.col("indice_selic"), f.lag("indice_selic", 21).over(window_spec)) - 1
    ) \
    .withColumn(
        "ipca_acum_21d", 
        f.try_divide(f.col("indice_ipca"), f.lag("indice_ipca", 21).over(window_spec)) - 1
    ) \
    .withColumn(
        "ibov_retorno_21d", 
        f.try_divide(f.col("ibov_close"), f.lag("ibov_close", 21).over(window_spec)) - 1
    )\
    .withColumn(
        "cdi_acum_63d", 
        f.try_divide(f.col("indice_cdi"), f.lag("indice_cdi", 63).over(window_spec)) - 1
    ) \
    .withColumn(
        "selic_acum_63d", 
        f.try_divide(f.col("indice_selic"), f.lag("indice_selic", 63).over(window_spec)) - 1
    ) \
    .withColumn(
        "ipca_acum_63d", 
        f.try_divide(f.col("indice_ipca"), f.lag("indice_ipca", 63).over(window_spec)) - 1
    ) \
    .withColumn(
        "ibov_retorno_63d", 
        f.try_divide(f.col("ibov_close"), f.lag("ibov_close", 63).over(window_spec)) - 1
    ) \
    .withColumn(
        "cdi_acum_126d", 
        f.try_divide(f.col("indice_cdi"), f.lag("indice_cdi", 126).over(window_spec)) - 1
    ) \
    .withColumn(
        "selic_acum_126d", 
        f.try_divide(f.col("indice_selic"), f.lag("indice_selic", 126).over(window_spec)) - 1
    ) \
    .withColumn(
        "ipca_acum_126d", 
        f.try_divide(f.col("indice_ipca"), f.lag("indice_ipca", 126).over(window_spec)) - 1
    ) \
    .withColumn(
        "ibov_retorno_126d", 
        f.try_divide(f.col("ibov_close"), f.lag("ibov_close", 126).over(window_spec)) - 1
    ) \
    .withColumn(
        "cdi_acum_252d",
        f.try_divide(f.col("indice_cdi"), f.lag("indice_cdi", 252).over(window_spec)) - 1
    ) \
    .withColumn(
        "selic_acum_252d", 
        f.try_divide(f.col("indice_selic"), f.lag("indice_selic", 252).over(window_spec)) - 1
    ) \
    .withColumn(
        "ipca_acum_252d", 
        f.try_divide(f.col("indice_ipca"), f.lag("indice_ipca", 252).over(window_spec)) - 1
    ) \
    .withColumn(
        "ibov_retorno_252d", 
        f.try_divide(f.col("ibov_close"), f.lag("ibov_close", 252).over(window_spec)) - 1
    )\
    .withColumn(
        "retorno_benchmark_21d", # retorno do benchmark específico do fundo em 1 mês
        f.when(f.col("benchmark_normalizado") == "CDI", f.col("cdi_acum_21d"))
         .when(f.col("benchmark_normalizado") == "Selic", f.col("selic_acum_21d"))
         .when(f.col("benchmark_normalizado") == "IPCA", f.col("ipca_acum_21d"))
         .when(f.col("benchmark_normalizado") == "Ibovespa", f.col("ibov_retorno_21d"))
    )\
    .withColumn(
        "retorno_benchmark_63d", # retorno do benchmark específico do fundo em 3 mês
        f.when(f.col("benchmark_normalizado") == "CDI", f.col("cdi_acum_63d"))
         .when(f.col("benchmark_normalizado") == "Selic", f.col("selic_acum_63d"))
         .when(f.col("benchmark_normalizado") == "IPCA", f.col("ipca_acum_63d"))
         .when(f.col("benchmark_normalizado") == "Ibovespa", f.col("ibov_retorno_63d"))
    )\
    .withColumn(
        "retorno_benchmark_126d", # retorno do benchmark específico do fundo em 6 mês
        f.when(f.col("benchmark_normalizado") == "CDI", f.col("cdi_acum_126d"))
         .when(f.col("benchmark_normalizado") == "Selic", f.col("selic_acum_126d"))
         .when(f.col("benchmark_normalizado") == "IPCA", f.col("ipca_acum_126d"))
         .when(f.col("benchmark_normalizado") == "Ibovespa", f.col("ibov_retorno_126d"))
    )\
    .withColumn(
        "retorno_benchmark_252d", # retorno do benchmark do fundo em 1 ano
        f.when(f.col("benchmark_normalizado") == "CDI", f.col("cdi_acum_252d"))
         .when(f.col("benchmark_normalizado") == "Selic", f.col("selic_acum_252d"))
         .when(f.col("benchmark_normalizado") == "IPCA", f.col("ipca_acum_252d"))
         .when(f.col("benchmark_normalizado") == "Ibovespa", f.col("ibov_retorno_252d"))
    )\
    .withColumn(
        "alpha_21d", 
        f.col("retorno_21d") - f.col("retorno_benchmark_21d") # quanto o fundo superou o benchmark em 1 mes
    )\
    .withColumn(
        "alpha_63d", 
        f.col("retorno_63d") - f.col("retorno_benchmark_63d") # quanto o fundo superou o benchmark em 3 mes
    )\
    .withColumn(
        "alpha_126d", 
        f.col("retorno_126d") - f.col("retorno_benchmark_126d") # quanto o fundo superou o benchmark em 6 mes
    )\
    .withColumn(
        "alpha_252d", 
        f.col("retorno_252d") - f.col("retorno_benchmark_252d") # quanto o fundo superou o benchmark em 1 ano
    )\
    .withColumn(
        "sharpe_252d", 
        f.try_divide(f.col("retorno_252d") - f.col("selic_acum_252d"), f.col("volatilidade_252d"))
    )\
    .withColumn(
        "sortino_252d", 
        f.try_divide(f.col("retorno_252d") - f.col("selic_acum_252d"), f.col("downside_deviation_252d"))
    )\
    .withColumn(
        "ano_mes", 
        f.date_format(f.col("dt_comptc"), "yyyy-MM") # Gera o formato "2026-03"
    )

### 6. Drop de Colunas

In [0]:
gold_fato_diario = gold_fato_diario.drop("data", "data_processamento", "data_inicio", "cnpj_classe", "tipo_classe")

## CAMADA TESTE DE VALOR DA COTA

In [0]:
df_teste = gold_fato_diario.filter(f.col("cnpj_fundo_classe") == "970059000102")\
  .select("dt_comptc", "vl_quota", "retorno_252d", "retorno_benchmark_252d","benchmark_normalizado")\
  .orderBy("dt_comptc") 

In [0]:
display(df_teste)

### 7. Salvar os dados

In [0]:
# gold_fato_diario = gold_fato_diario.withColumn(
#     "data_processamento",
#     f.date_format(f.current_date(), "yyyyMMdd").cast("int")
# )



# data_proc = int(datetime.now().strftime(f"%Y%m%d"))

# gold_fato_diario.write \
#     .mode('overwrite') \
#     .partitionBy("data_processamento") \
#     .option("replaceWhere", f"data_processamento = {data_proc}")\
#     .format('delta')\
#     .saveAsTable("workspace.case_spark_cvm.gold_fato_diario")

In [0]:
display(gold_fato_diario)